# requires-grad-propagation — faded example 1: Implement the any-Input Gate in requires_grad Propagation (Faded)

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-propagation`. The last cell reports your progress on the `Backprop: requires_grad propagation` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: requires_grad propagation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`requires-grad-propagation`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "requires-grad-propagation"
DD_SUBTOPIC = "Backprop: requires_grad propagation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The third gate in the `requires_grad` propagation rule checks whether any input tensor has `requires_grad=True`. The scan must filter out non-tensor args (Python ints, floats, tuples) using `isinstance(a, torch.Tensor)` to avoid `AttributeError`. The full three-gate expression is `grad_tracking_enabled AND is_differentiable AND any(isinstance(a, Tensor) and a.requires_grad for a in args)`.

## Faded exercise 1

Implement `propagate_requires_grad(args, is_differentiable, grad_tracking_enabled)`. The `grad_tracking_enabled and is_differentiable` part is provided. Your task is to **compute the `any_tracked` expression** that scans `args` for Tensor inputs with `requires_grad=True`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def propagate_requires_grad(
    args: tuple,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    any_tracked = any(isinstance(a, t.Tensor) and a.requires_grad for a in args)
    return grad_tracking_enabled and is_differentiable and any_tracked


def _test():
    import torch as t
    leaf = t.tensor([1.0], requires_grad=True)
    const = t.tensor([2.0])
    # All three gates True -> True
    assert propagate_requires_grad((leaf, const, 3.0), True, True) == True
    # Tracking off -> False
    assert propagate_requires_grad((leaf, const), True, False) == False
    # No tracked input -> False
    assert propagate_requires_grad((const, 5.0, (1, 2)), True, True) == False
    # Non-tensor args should not crash
    assert propagate_requires_grad((leaf, 42, None, (1,2,3)), True, True) == True


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def propagate_requires_grad(
    args: tuple,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    any_tracked = any(isinstance(a, t.Tensor) and a.requires_grad for a in args)
    return grad_tracking_enabled and is_differentiable and any_tracked
```
</details>